In [ ]:
"""
Elasticsearch index mapping and query execution notebook.
"""

import json

import requests
from elastic_config import BASE_URL, CA_CERT_PATH, PASSWORD, USERNAME

%load_ext autoreload
%autoreload 2

## Searching in `search_as_you_type` and its subfields

In [ ]:
# Queries the tech_books4 index in Elasticsearch using credentials from a .env file.
url = f"{BASE_URL}/tech_books4/_search"

payload = {
    "query": {
        "multi_match": {
            "query": "in",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    },
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=payload
)

print(json.dumps(response.json(), indent=2))

## Scenario 1: The Raw Prefix Match

In [ ]:
# What it tests: How _index_prefix catches unfinished words right at the beginning.
url = f"{BASE_URL}/tech_books4/_search"

payload = {
    "query": {
        "multi_match": {
            "query": "elast",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    },
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=payload
)

print(json.dumps(response.json(), indent=2))

## Scenario2: The Space Trigger (The Eliminator)

In [ ]:
# What it tests: How the engines uses spaces to declare a word "completed".
url = f"{BASE_URL}/tech_books4/_search"

payload = {
    "query": {
        "multi_match": {
            "query": "elastic ",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    },
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=payload
)

print(json.dumps(response.json(), indent=2))

## Scenario 3: The 2-Gram Scoring Test

In [ ]:
# What it tests: How the engines splits completed words for exact phrase matching (_2gram)
# while still applying the prefix to the last letter.
url = f"{BASE_URL}/tech_books4/_search"

payload = {
    "query": {
        "multi_match": {
            "query": "Elasticsearch i",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
            "operator": "AND",
        }
    },
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=payload
)

print(json.dumps(response.json(), indent=2))

## Scenario 4: The Middle-of-String Match

In [ ]:
# A common misconception is that autocomplete only works from the very first letter of a title
url = f"{BASE_URL}/tech_books4/_search"

payload = {
    "query": {
        "multi_match": {
            "query": "in a",
            "type": "bool_prefix",
            "fields": ["title", "title._2gram", "title._3gram"],
        }
    },
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=payload
)

print(json.dumps(response.json(), indent=2))

## The Query to Force the Scan

In [ ]:
# Do not apply any smart optimizations.
# Go directly to the title._2gram field and mathematically scan it
# for anything starting with 'elasticsearch i'.
url = f"{BASE_URL}/tech_books4/_search"

prefix_payload: dict = {
    "explain": True,
    "query": {"prefix": {"title._2gram": {"value": "elasticsearch i"}}},
}

response = requests.post(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=prefix_payload
)

print(json.dumps(response.json(), indent=2))

## Fields with multiple data types

In [ ]:
# Fields with multiple data types
# create multiple fields with subject of type text, subject.kw of type keyword
url = f"{BASE_URL}/my_email"

mapping_payload: dict = {
    "mappings": {
        "properties": {
            "subject": {"type": "text", "fields": {"kw": {"type": "keyword"}}}
        }
    }
}

response = requests.put(
    url=url, auth=(USERNAME, PASSWORD), verify=CA_CERT_PATH, json=mapping_payload
)

print(json.dumps(response.json(), indent=2))